In [1]:
from pathlib import Path
import json
import math
import shutil

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2Config,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

## paths

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data"
outputs_dir = project_root / "outputs"

text_jsonl = data_dir / "punjabi_corpus_cleaned.jsonl"
tokenizer_dir = outputs_dir / "punjabi_tokenizer_16k"

run_name = "punjabi-gpt-scratch-30m"
model_output_dir = outputs_dir / run_name
final_model_dir = outputs_dir / f"{run_name}-final"

print("JSONL exists:", text_jsonl.exists(), text_jsonl)
print("Tokenizer exists:", tokenizer_dir.exists(), tokenizer_dir)
print("Output dir:", model_output_dir)

JSONL exists: True /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/data/punjabi_corpus_cleaned.jsonl
Tokenizer exists: True /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi_tokenizer_16k
Output dir: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi-gpt-scratch-30m


## device check

In [3]:
print("Torch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Torch version: 2.10.0
MPS available: True
MPS built: True
Using device: mps


## load tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_dir))

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer vocab size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer vocab size: 16000
Pad token: <|endoftext|>
EOS token: <|endoftext|>


## load dataset

In [5]:
dataset = load_dataset(
    "json",
    data_files={"train": str(text_jsonl)},
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 193009
    })
})

## optional validation split

In [6]:
split_dataset = dataset["train"].train_test_split(test_size=0.02, seed=42)

train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]

print("Train rows:", len(train_dataset))
print("Valid rows:", len(valid_dataset))

Train rows: 189148
Valid rows: 3861


## tokenization settings

In [7]:
max_length = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_special_tokens_mask=True,
    )

## tokenize dataset

In [8]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_train)
print(tokenized_valid)

Map:   0%|          | 0/189148 [00:00<?, ? examples/s]

Map:   0%|          | 0/3861 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 189148
})
Dataset({
    features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 3861
})


## model config

In [9]:
config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=max_length,
    n_ctx=max_length,
    n_embd=384,
    n_layer=8,
    n_head=6,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = GPT2LMHeadModel(config)
model.to(device)

print(model)
print("Number of parameters:", f"{model.num_parameters():,}")

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(16000, 384)
    (wpe): Embedding(128, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPT2Block(
        (ln_1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=1152, nx=384)
          (c_proj): Conv1D(nf=384, nx=384)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=1536, nx=384)
          (c_proj): Conv1D(nf=384, nx=1536)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=384, out_features=16000, bias=False)
)
Number of parameters: 20,389,632


## data collector

In [15]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## clean model output dir if needed

In [16]:
if model_output_dir.exists():
    shutil.rmtree(model_output_dir)

model_output_dir.mkdir(parents=True, exist_ok=True)
print("Fresh output dir ready:", model_output_dir)

Fresh output dir ready: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi-gpt-scratch-30m


## training arguments

In [17]:
training_args = TrainingArguments(
    output_dir=str(model_output_dir),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,

    save_total_limit=2,
    report_to="none",

    fp16=False,
    dataloader_pin_memory=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## trainer

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

## train 

In [19]:
train_result = trainer.train()
train_result

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.615338,1.561607
2,1.531799,1.416886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189148, training_loss=1.7447409525251782, metrics={'train_runtime': 9431.2396, 'train_samples_per_second': 40.111, 'train_steps_per_second': 20.055, 'total_flos': 4124522187325440.0, 'train_loss': 1.7447409525251782, 'epoch': 2.0})

## evaluate

In [20]:
eval_results = trainer.evaluate()
eval_results

RuntimeError: on_train_begin must be called before on_evaluate

## perplexity

In [21]:
try:
    perplexity = math.exp(eval_results["eval_loss"])
except OverflowError:
    perplexity = float("inf")

print("Validation loss:", eval_results["eval_loss"])
print("Perplexity:", perplexity)

NameError: name 'eval_results' is not defined

## save final model

In [22]:
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

print("Saved final model to:", final_model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi-gpt-scratch-30m-final


## reload for inference

In [23]:
infer_tokenizer = AutoTokenizer.from_pretrained(str(final_model_dir))
infer_model = GPT2LMHeadModel.from_pretrained(str(final_model_dir)).to(device)

print("Loaded model from:", final_model_dir)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loaded model from: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi-gpt-scratch-30m-final


## punjabi generation helper

In [24]:
def generate_punjabi(prompt, max_new_tokens=80, temperature=0.8, top_p=0.95):
    inputs = infer_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=infer_tokenizer.eos_token_id
        )

    return infer_tokenizer.decode(outputs[0], skip_special_tokens=True)

## test prompt

In [25]:
prompts = [
    "ਪੰਜਾਬੀ ਭਾਸ਼ਾ",
    "ਇੱਕ ਪਿੰਡ ਵਿੱਚ",
    "ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ",
    "ਕਿਸਾਨ ਲਈ ਸਭ ਤੋਂ ਮਹੱਤਵਪੂਰਨ ਗੱਲ"
]

for p in prompts:
    print("=" * 80)
    print("PROMPT:", p)
    print(generate_punjabi(p, max_new_tokens=100))
    print()

PROMPT: ਪੰਜਾਬੀ ਭਾਸ਼ਾ
ਪੰਜਾਬੀ ਭਾਸ਼ਾਵਾਂ

ਪੰਜਾਬੀ ਭਾਸ਼ਾਵਾਂ (ਜਾਂ ਪੰਜਾਬੀਃ ਪੰਜਾਬੀ ਭਾਸ਼ਾਵਾਂ ਦਾ ਹਵਾ) ਇੱਕ ਭਾਰਤੀ ਵਿਗਿਆਨੀ ਹੈ ਜਿਸ ਨੂੰ ਆਮ ਤੌਰ ਉੱਤੇ ਮਨੁੱਖੀ ਵਿਗਿਆਨੀਆਂ ਦੁਆਰਾ ਰੱਖਿਆ ਅਤੇ ਜ਼ਿਆਦਾਤਰ ਪ੍ਰਸ

PROMPT: ਇੱਕ ਪਿੰਡ ਵਿੱਚ
ਇੱਕ ਪਿੰਡ ਵਿੱਚ ਪ੍ਰੋਜੈਕਟ

ਇੱਕ ਪਿੰਡ ਵਿੱਚ ਪ੍ਰੋਜੈਕਟ ਇੱਕ ਲੰਬੀ ਸ਼ੁਰੂਆਤੀ ਸ਼ੁਰੂਆਤੀ ਕਾਰਵਾਈ ਹੈ।
ਇਤਿਹਾਸ.
ਇਸ ਦੀ ਸਥਾਪਨਾ ਸੰਨ 1971 ਵਿੱਚ ਪਿੰਡ, ਕਾਰਵਾਈ ਅਤੇ ਸਾਹਿਤ ਨਿਰਮਾਣ ਅਤੇ ਕੋਲੰ

PROMPT: ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ
ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ

ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ ਦੇਸ਼ ਦੀਆਂ ਪ੍ਰਮੁੱਖ ਸਿੱਖਿਆਵਾਂ ਦੇ ਕਾਰਜਾਂ ਦੀ ਸਿੱਖਿਆ ਹੈ। ਇੱਕ ਵਿਸ਼ੇਸ਼ ਸਿੱਖਿਆ ਦੇ ਰੂਪ ਵਿੱਚ, ਇਹ ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਹਾਇਤਾ ਕਰਨ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਹੈ,

PROMPT: ਕਿਸਾਨ ਲਈ ਸਭ ਤੋਂ ਮਹੱਤਵਪੂਰਨ ਗੱਲ
ਕਿਸਾਨ ਲਈ ਸਭ ਤੋਂ ਮਹੱਤਵਪੂਰਨ ਗੱਲ

ਕਿਸਾਨ ਲਈ ਸਭ ਤੋਂ ਮਹੱਤਵਪੂਰਨ ਗੱਲ ਇੱਕ ਕਿਸਮ ਦਾ ਸਮੂਹ ਹੈ ਜੋ ਰੱਖਿਆਤਮਕ ਮੁਕਾਬਲਿਆਂ ਵਿੱਚ ਕੰਮ ਕਰਦਾ ਹੈ। ਇਹ ਮੁਕਾਬਲਾ ਸੈਨ ਫਰਾਂਸਿਸਕੋ ਵਿੱਚ ਕਿਸਾਨ ਦੇ ਚੋਟੀ ਦੇ ਕਿਸੇ ਵੀ ਦੁਆਲੇ ਦ



## save sample geenration

In [26]:
samples_path = outputs_dir / "scratch_model_generations.txt"

with open(samples_path, "w", encoding="utf-8") as f:
    for p in prompts:
        out = generate_punjabi(p, max_new_tokens=100)
        f.write(f"PROMPT: {p}\n")
        f.write(f"OUTPUT: {out}\n")
        f.write("=" * 100 + "\n")

print("Saved generations to:", samples_path)

Saved generations to: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/scratch_model_generations.txt


In [27]:
import math

eval_loss = 1.416886
perplexity = math.exp(eval_loss)

print("Validation loss:", eval_loss)
print("Perplexity:", perplexity)

Validation loss: 1.416886
Perplexity: 4.124257485215774


## import saved model and run 

In [30]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

# Paths
project_root = Path("/Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt")
model_dir = project_root / "outputs" / "punjabi-gpt-scratch-30m-final"

# Device
device = "mps" if torch.backends.mps.is_available() else "cpu"

# Load
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = GPT2LMHeadModel.from_pretrained(str(model_dir)).to(device)
model.eval()

def generate_text(prompt, max_new_tokens=500, temperature=0.8, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:

# Example
prompt = "ਪੰਜਾਬੀ ਭਾਸ਼ਾ"
result = generate_text(prompt)
print(result)

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (128). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


ਪੰਜਾਬੀ ਭਾਸ਼ਾ

ਪੰਜਾਬੀ ਭਾਸ਼ਾ ਭਾਸ਼ਾ ਭਾਸ਼ਾ ਦੇ ਕਾਨੂੰਨ ਅਤੇ ਰਾਸ਼ਟਰੀ ਵਿਗਿਆਨੀਆਂ ਲਈ ਸੰਗਠਨ ਦਾ ਇੱਕ ਸਮੂਹ ਹੈ। ਕਾਨੂੰਨੀ ਸ਼ਬਦਾਂ ਵਿੱਚ, ਸ਼ਾਸਨ ਦੀ ਪ੍ਰਕਿਰਿਆ ਸ਼ਾਮਲ ਹੁੰਦੀ ਹੈ, ਪੰਜਾਬੀ ਭਾਸ਼ਾ ਸਿਧਾਂਤਾਂ, ਪਾਲੀਆਂ ਅਤੇ ਮੂਨੀ ਭਾਲਿਮਾਸਿਸ਼ਾਵਾਹ ਹਨ। ਉਹ ਪੰਜਾਲਿਆਂ ਦੇ ਦੀ ਹੋਮਾਲੇ ਪੰਜਾਲਾਲੀ ਮੰਗਾਲੇਜਾਲ ਦੀ ਨੂੰਹ ਦੇਜਾਲੀ ਦੇ ਘਰ ਵਿੱਲੇ ਲੋ (ਡ਼ੀ ਜਾਂ ਦੀ ਹੋਲ ਵਿਆਂ ਦੇ ਦਾਲੇ ਸ਼ਮ) ਵਿੱਚੇ ਪਾਲੀਆਂ ਵਿੱਲਾਲੂਲਾਲੂਨੀ ਹੋਕਾਲ ਦੇ ਵਿੱਲੀ ਜਾਂ ਦਾਲੇ ਸਥਿਤਾਂ ਦੇ ਹੋਰੀ ਭਾਲਾਲੇ ਹੁੰਦੀ ਹੈ। ਜਦੋਂ ਇੱਤੇ ਵਾਲੀ ਹੁੰਦੀਮਾਂ ਦੇ ਦੀਨੀ ਹੋਰੀ ਹੋ ਪਾਲਾਲੇ ਅਧਿਖ ਵਿਆਂਦਾਲਿਨੀ ਹੁੰਦੇ ਪੁਨੀਆਂਦੀਆਂਦੀ ਹੋਸ਼ਾ ਹਨ। ਉਹ ਕਈ ਵਿੱਲੇ ਹੁੰਦੀਆਂਦੇ ਵਾਲ ਹੁੰਦੀਆਂ ਹਨ, ਜਾਲੇ ਸਾਸ਼ਿਪਾਲੀ ਹੁੰਦੀ ਵਿੱਚ ਕਾਲਾਂ ਦੇ ਪੰਡੀਆਂ ਦੀਆਂਦੀਆਂ ਵਿੱਚ ਦੇ ਨੂੰਹ ਦੇ ਦੇ ਦਾਲੇ ਦੁਕਾਲੀਆਂਦੀਆਂਦੇ ਅੱਖਣ ਤੋਂਦੇ ਇੱਤਾਂ ਦੀ ਹੁੰਦੀ ਹੋ ਲਾਸ਼ਾ ਪੰਦੇ ਭਾਲ ਹਨ। ਉਹ ਕਾ ਹੁੰਦੇ ਪਾਲਾਲਾਲ ਪ੍ਰ ਹੁੰਦੀ ਹੁੰਦਾ ਇੱਟ ਸ
